In [3]:
import os
import glob
import pandas as pd

# Setup paths relative to scripts/preprocessing/
CURRENT_DIR = os.getcwd()
BASE_DIR = os.path.abspath(os.path.join(CURRENT_DIR, "..", ".."))
DATA_DIR = os.path.join(BASE_DIR, "data_log", "data_eish")

# Find all raw CSV log files
raw_csv_files = glob.glob(os.path.join(DATA_DIR, "imu_log_*.csv"))

# Dictionary to hold all loaded dataframes: { filename: dataframe }
imu_data = {}

for file_path in raw_csv_files:
    filename = os.path.basename(file_path)
    
    # Skip empty 0-byte files
    if os.path.getsize(file_path) > 0:
        imu_data[filename] = pd.read_csv(file_path)
        print(f"✅ Loaded: {filename} ({len(imu_data[filename])} rows)")
    else:
        print(f"⚠️ Skipped empty file: {filename}")

print(f"\n🎉 Total raw datasets loaded: {len(imu_data)}")

✅ Loaded: imu_log_20260721_032000.csv (3254 rows)
✅ Loaded: imu_log_20260721_032000_cleaned.csv (1582 rows)
✅ Loaded: imu_log_20260722_234935.csv (8372 rows)
✅ Loaded: imu_log_20260722_234935_cleaned.csv (3563 rows)
✅ Loaded: imu_log_20260727_121550.csv (4701 rows)
✅ Loaded: imu_log_20260727_121550_cleaned.csv (1353 rows)
⚠️ Skipped empty file: imu_log_20260727_123141.csv

🎉 Total raw datasets loaded: 6


In [6]:
# cleaning the dataframes by removing rows with NaN values
for filename, df in imu_data.items():
    initial_row_count = len(df)
    df.dropna(inplace=True)
    cleaned_row_count = len(df)
    print(f"🧹 Cleaned: {filename} | Removed {initial_row_count - cleaned_row_count} rows with NaN values | Remaining rows: {cleaned_row_count}")
    

🧹 Cleaned: imu_log_20260721_032000.csv | Removed 0 rows with NaN values | Remaining rows: 3254
🧹 Cleaned: imu_log_20260721_032000_cleaned.csv | Removed 0 rows with NaN values | Remaining rows: 1582
🧹 Cleaned: imu_log_20260722_234935.csv | Removed 0 rows with NaN values | Remaining rows: 8372
🧹 Cleaned: imu_log_20260722_234935_cleaned.csv | Removed 0 rows with NaN values | Remaining rows: 3563
🧹 Cleaned: imu_log_20260727_121550.csv | Removed 0 rows with NaN values | Remaining rows: 4701
🧹 Cleaned: imu_log_20260727_121550_cleaned.csv | Removed 0 rows with NaN values | Remaining rows: 1353


In [12]:
import pandas as pd

for filename, df in imu_data.items():
    # Safety Check 1: Skip if DataFrame is completely empty or None
    if df is None or df.empty:
        print(f"⚠️ Skipped {filename}: DataFrame is empty.")
        continue

    # Clean whitespace from column names (e.g., 'timestamp ' -> 'timestamp')
    df.columns = df.columns.str.strip()

    # Safety Check 2: Match 'timestamp' regardless of capitalization (Timestamp, timestamp, etc.)
    matching_cols = [c for c in df.columns if c.lower() == 'timestamp']

    if not matching_cols:
        print(f"⚠️ Skipped {filename}: No 'timestamp' column found. Columns present: {list(df.columns)}")
        continue

    col_name = matching_cols[0]
    initial_row_count = len(df)

    # Make an explicit copy to prevent SettingWithCopyWarning
    df_clean = df.copy()

    # 1. Convert to datetime
    df_clean[col_name] = pd.to_datetime(df_clean[col_name], errors='coerce')

    # 2. Drop rows with invalid timestamps (NaT)
    df_clean = df_clean.dropna(subset=[col_name])

    # 3. Sort chronologically
    df_clean = df_clean.sort_values(by=col_name).reset_index(drop=True)

    # 4. Save cleaned copy back to dictionary
    imu_data[filename] = df_clean

    cleaned_row_count = len(df_clean)
    removed_rows = initial_row_count - cleaned_row_count

    print(f"⏱️ Processed: {filename} | Removed {removed_rows} invalid rows | Remaining: {cleaned_row_count}")

⏱️ Processed: imu_log_20260721_032000.csv | Removed 0 invalid rows | Remaining: 3254
⏱️ Processed: imu_log_20260721_032000_cleaned.csv | Removed 0 invalid rows | Remaining: 1582
⏱️ Processed: imu_log_20260722_234935.csv | Removed 0 invalid rows | Remaining: 8372
⏱️ Processed: imu_log_20260722_234935_cleaned.csv | Removed 0 invalid rows | Remaining: 3563
⏱️ Processed: imu_log_20260727_121550.csv | Removed 0 invalid rows | Remaining: 4701
⏱️ Processed: imu_log_20260727_121550_cleaned.csv | Removed 0 invalid rows | Remaining: 1353


In [13]:
#fixed timestamp column name to 'timestamp' for consistency
for filename, df in imu_data.items():
    if 'Timestamp' in df.columns:
        df.rename(columns={'Timestamp': 'timestamp'}, inplace=True)
#resampling the dataframes to 100Hz
for filename, df in imu_data.items():
    if 'timestamp' not in df.columns:
        print(f"⚠️ Skipped {filename}: No 'timestamp' column found for resampling.")
        continue

    # Set timestamp as index
    df.set_index('timestamp', inplace=True)

    # Resample to 100Hz (10ms intervals)
    df_resampled = df.resample('10ms').mean().interpolate()

    # Reset index to bring timestamp back as a column
    df_resampled.reset_index(inplace=True)

    # Save resampled dataframe back to dictionary
    imu_data[filename] = df_resampled

    print(f"🔄 Resampled: {filename} | New row count: {len(df_resampled)}")
    

🔄 Resampled: imu_log_20260721_032000.csv | New row count: 7345
🔄 Resampled: imu_log_20260721_032000_cleaned.csv | New row count: 7345
🔄 Resampled: imu_log_20260722_234935.csv | New row count: 19461
🔄 Resampled: imu_log_20260722_234935_cleaned.csv | New row count: 19461
🔄 Resampled: imu_log_20260727_121550.csv | New row count: 12289
🔄 Resampled: imu_log_20260727_121550_cleaned.csv | New row count: 12289


In [14]:
#removing duplicate rows based on timestamp
for filename, df in imu_data.items():
    if 'timestamp' not in df.columns:
        print(f"⚠️ Skipped {filename}: No 'timestamp' column found for duplicate removal.")
        continue

    initial_row_count = len(df)
    df.drop_duplicates(subset='timestamp', keep='first', inplace=True)
    cleaned_row_count = len(df)

    imu_data[filename] = df  # Save back to dictionary

    print(f"🗑️ Duplicates removed: {filename} | Removed {initial_row_count - cleaned_row_count} rows | Remaining: {cleaned_row_count}")

🗑️ Duplicates removed: imu_log_20260721_032000.csv | Removed 0 rows | Remaining: 7345
🗑️ Duplicates removed: imu_log_20260721_032000_cleaned.csv | Removed 0 rows | Remaining: 7345
🗑️ Duplicates removed: imu_log_20260722_234935.csv | Removed 0 rows | Remaining: 19461
🗑️ Duplicates removed: imu_log_20260722_234935_cleaned.csv | Removed 0 rows | Remaining: 19461
🗑️ Duplicates removed: imu_log_20260727_121550.csv | Removed 0 rows | Remaining: 12289
🗑️ Duplicates removed: imu_log_20260727_121550_cleaned.csv | Removed 0 rows | Remaining: 12289


In [16]:
print(df.head())  # Display the first few rows of the last processed DataFrame for verification

                timestamp      AccX  AccY      AccZ      GyroX      GyroY  \
0 2026-07-27 12:15:56.150  0.190000  1.00  1.200000  15.260000 -72.750000   
1 2026-07-27 12:15:56.160  0.146667  0.93  1.171667  19.338333 -69.281667   
2 2026-07-27 12:15:56.170  0.103333  0.86  1.143333  23.416667 -65.813333   
3 2026-07-27 12:15:56.180  0.060000  0.79  1.115000  27.495000 -62.345000   
4 2026-07-27 12:15:56.190  0.016667  0.72  1.086667  31.573333 -58.876667   

       GyroZ  AccX_smooth  AccY_smooth  AccZ_smooth  GyroX_smooth  \
0  52.670000     0.190000        1.000     1.200000     15.260000   
1  45.986667     0.168333        0.965     1.185833     17.299167   
2  39.303333     0.146667        0.930     1.171667     19.338333   
3  32.620000     0.125000        0.895     1.157500     21.377500   
4  25.936667     0.103333        0.860     1.143333     23.416667   

   GyroY_smooth  GyroZ_smooth  
0    -72.750000     52.670000  
1    -71.015833     49.328333  
2    -69.281667     45.986

In [17]:
# Save the loaded dataframes to a single pickle file for later use
pickle_file_path = os.path.join(DATA_DIR, "imu_data.pkl")